In [2]:
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from itertools import islice

g:\NeuroMed-VLM\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset(
    "eltorio/ROCOv2-radiology",
    streaming = True,
    split = "train"
)

dataset_shuffle = dataset.shuffle(seed = 42, buffer_size = 5000)
samples = list(islice(dataset_shuffle,1000))

In [9]:
class MedicalDataset(Dataset):
    def __init__(self,samples,transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return(len(self.samples))

    def __getitem__(self,index):
        sample = self.samples[index]
        return {
            "image": self.transform(sample["image"].convert("RGB")),
            "caption": sample["caption"],
            "image_id": sample["image_id"]
        }


In [ ]:
transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

dataset = MedicalDataset(samples, transform=transform)

sample = dataset[0]


[23, 25, 18, 12]


In [28]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")

def collate_fn(batch,tokenizer):
    images = [sample["image"] for sample in batch]
    captions = [sample["caption"] for sample in batch]
    image_ids = [sample["image_id"] for sample in batch]
    image_batch = torch.stack(images,dim = 0)
    encoded = tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    return{ "pixel_values": image_batch,
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "image_ids": image_ids}


In [ ]:
test_batch = [dataset[i] for i in range(4)]
result = collate_fn(test_batch,tokenizer)

print("Images:", result["pixel_values"].shape)
print("Text:", result["input_ids"].shape)
print("Mask:", result["attention_mask"].shape)
print("IDs:", len(result["image_ids"]))

Images: torch.Size([4, 3, 224, 224])
Text: torch.Size([4, 43])
Mask: torch.Size([4, 43])
IDs: 4


In [32]:
from torch.utils.data import DataLoader
from functools import partial

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=partial(collate_fn, tokenizer=tokenizer)
)

first_batch = next(iter(loader))

print(first_batch["pixel_values"].shape)
print(first_batch["input_ids"].shape)

torch.Size([4, 3, 224, 224])
torch.Size([4, 51])


In [ ]:
num_batches = 0
num_samples = 0

for batch in loader:

    images = batch["pixel_values"]
    input_ids = batch["input_ids"]

    assert images.shape[1:] == (3, 224, 224)
    assert images.shape[0] == input_ids.shape[0]
    assert input_ids.shape[1] <= 128

    num_batches += 1
    num_samples += 4

print("Total batches:", num_batches)
print("Total samples:", num_samples)

Total batches: 250
Total samples: 1000
